
# 24 — EWM state trajectory ↔ stabilizer formalism (GF(2) shadow)

Notebook 23 turned the ring + lattice into tools. This one asks how far those
tools reach into quantum mechanics. The honest answer:

> The EWM ring is the **GF(2) / stabilizer (Clifford–Gaussian) shadow** of a
> quantum state trajectory — the same linear algebra — but *not* a quantum
> state. Rank = entropy, change-of-basis = Clifford operation, and the code
> filtration are exact; amplitudes, phases, interference and orthomodularity
> are not.

| EWM | stabilizer / Gaussian formalism |
| - | - |
| HLLSet bit vector | Pauli operator as a binary vector in `GF(2)^{2n}` (X-block, Z-block) |
| `Δ` (XOR) | group multiplication mod 2 |
| span of originals | binary linear code / stabilizer group exponent |
| `dimension()` | code rank; stabilizer-state entanglement entropy |
| RREF basis + pivots | generator matrix + Gaussian elimination |
| `change_of_basis` (pure re-basis) | invertible GF(2) transform ⇒ **Clifford** frame change |
| span extension | adding a stabilizer generator (isometry, not invertible) |
| windowed eviction | discarding a generator (irreversible) |
| monotone basis | a **filtration of codes** `S₁ ⊆ S₂ ⊆ …` |
| `cover` | classical support of the code |
| `residual ≠ 0` | off-code component — analogue of non-stabilizerness ("magic") |
| meet table | subgroup-intersection data (a group closes under `·`, not `∩`) |

What we verify numerically below:

- §1 the symplectic form: stabilizer generators must pairwise **commute**;
- §2 **rank = entropy** on graph states (numeric partial trace vs GF(2) rank);
- §3 stabilizer **overlap** is governed by the **group intersection**, and
  where the first-order (bitwise) measure differs;
- §4 the EWM trajectory as a **code filtration**, with Clifford-change vs
  extension vs eviction;
- §5 **where it breaks**: the HLLSet lattice is Boolean (distributive), the
  quantum projection lattice is not.


## §0 — Pauli vectors, the symplectic form, GF(2) rank

A Pauli operator is represented as `(p, x, z)` meaning `iᵖ Xˣ Zᶻ`, with `x, z`
bitmasks over `n` qubits. Multiplication and the commutator become GF(2)
linear algebra.

In [1]:

import itertools

import numpy as np

I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)


def pauli_mul(a, b, n):
    """(iᵖ Xˣ Zᶻ)(iᵖ' Xˣ' Zᶻ') — Pauli multiplication with phase."""
    p1, x1, z1 = a
    p2, x2, z2 = b
    sign = bin(z1 & x2).count("1") % 2        # (-1)^{z1·x2} = i^{2 z1·x2}
    return ((p1 + p2 + 2 * sign) % 4, x1 ^ x2, z1 ^ z2)


def pauli_trace(a, n):
    p, x, z = a
    return (1j ** p) * (2 ** n) if (x == 0 and z == 0) else 0j


def symplectic(a, b, n):
    """⟨a,b⟩ = x·z' + z·x' (mod 2); 0 ⇔ the Paulis commute."""
    _, x1, z1 = a
    _, x2, z2 = b
    return (bin(x1 & z2).count("1") + bin(z1 & x2).count("1")) % 2


def group_closure(gens, n):
    G = [(0, 0, 0)]
    for g in gens:
        G = G + [pauli_mul(g, h, n) for h in G]
    return list(dict.fromkeys(G))


def gf2_rank(rows, n):
    A = np.zeros((len(rows), n), dtype=np.uint8)
    for i, row in enumerate(rows):
        if np.isscalar(row):
            A[i] = [(int(row) >> j) & 1 for j in range(n)]
        else:
            A[i] = row
    m, r = len(rows), 0
    for c in range(n):
        p = next((i for i in range(r, m) if A[i, c]), None)
        if p is None:
            continue
        A[[r, p]] = A[[p, r]]
        for i in range(m):
            if i != r and A[i, c]:
                A[i] ^= A[r]
        r += 1
        if r == m:
            break
    return r


def gf2_intersection_dim(A, B, n):
    return gf2_rank(A, n) + gf2_rank(B, n) - gf2_rank(A + B, n)


## §1 — The symplectic form: stabilizer generators commute

A stabilizer group is an **abelian** subgroup with `-I ∉ G`. In vector terms
its generators are pairwise orthogonal under the symplectic form and span an
**isotropic** subspace. An anticommuting pair can never both be stabilizers of
one state: measuring one randomizes the other.

In [2]:

n = 2
X1 = (0, 0b01, 0b00)
Z1 = (0, 0b00, 0b01)
Z2 = (0, 0b00, 0b10)
X1Z2 = (0, 0b01, 0b10)

print("commute     (X1, X1Z2): symplectic =", symplectic(X1, X1Z2, n))
print("anticommute (Z1, X1Z2): symplectic =", symplectic(Z1, X1Z2, n))
print("anticommute (X1, Z1):   symplectic =", symplectic(X1, Z1, n))

# isotropic check: all pairwise symplectic products zero
gens_ok = [X1Z2, (0, 0b10, 0b01)]      # X1Z2 and Z1X2 commute
gens_bad = [X1, Z1]
for name, gens in [("valid stabilizer generators", gens_ok),
                   ("invalid (anticommuting)", gens_bad)]:
    pairs = list(itertools.combinations(gens, 2))
    iso = all(symplectic(a, b, n) == 0 for a, b in pairs)
    G = group_closure(gens, n)
    print("%-28s isotropic=%-5s |G|=%d, span dim=%d"
          % (name, iso, len(G), gf2_rank([x | (z << n) for _, x, z in gens], 2 * n)))

commute     (X1, X1Z2): symplectic = 0
anticommute (Z1, X1Z2): symplectic = 1
anticommute (X1, Z1):   symplectic = 1
valid stabilizer generators  isotropic=True  |G|=4, span dim=2
invalid (anticommuting)      isotropic=False |G|=4, span dim=2



## §2 — Rank = entropy (graph states)

For a stabilizer state, the entanglement entropy of a subsystem is the GF(2)
rank of a submatrix of the symplectic/adjacency matrix. On a graph state built
from a graph `Γ`, the entropy of the first `k` qubits is the rank of the
bipartite block `Γ[A, B]`. We check it against the numerically computed von
Neumann entropy of the reduced state (base-2, so it should equal the rank).

In [3]:

def graph_state(n, edges):
    N = 2 ** n
    idx = np.arange(N)
    bits = ((idx[:, None] >> np.arange(n - 1, -1, -1)) & 1).astype(np.int8)
    exp = np.zeros(N, dtype=np.int8)
    for (i, j) in edges:
        exp ^= (bits[:, i] & bits[:, j])
    return ((-1.0) ** exp) / np.sqrt(N)


def entanglement_entropy(psi, n, k):
    M = psi.reshape(2 ** k, 2 ** (n - k))
    rho = M @ M.conj().T
    ev = np.linalg.eigvalsh(rho)
    ev = ev[ev > 1e-12]
    return float(-np.sum(ev * np.log2(ev)))


n = 5
edges = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 4)]     # a 5-cycle
A = np.zeros((n, n), dtype=np.uint8)
for (i, j) in edges:
    A[i, j] = A[j, i] = 1
psi = graph_state(n, edges)

print("path/cycle graph state, n=%d, edges=%s" % (n, edges))
print("  k   S(von Neumann, bits)   rank(Gamma[A,B])")
for k in range(n + 1):
    S = entanglement_entropy(psi, n, k)
    blk = A[:k, k:]
    r = gf2_rank([row for row in blk], n - k) if k not in (0, n) else 0
    print("  %d   %8.4f                %d" % (k, S, r))

path/cycle graph state, n=5, edges=[(0, 1), (1, 2), (2, 3), (3, 4), (0, 4)]
  k   S(von Neumann, bits)   rank(Gamma[A,B])
  0     0.0000                0
  1     1.0000                1
  2     2.0000                2
  3     2.0000                2
  4     1.0000                1
  5    -0.0000                0



## §3 — Overlap of two stabilizer states: a **subgroup** meet

The overlap is fixed by the intersection of the two stabilizer groups. With
`ρ_G = (1/|G|) Σ_{g∈G} g` (rank-1 for a maximal isotropic `G`), `|G|=|H|=2ⁿ`:

```text
|⟨ψ_G|φ_H⟩|² = tr(ρ_G ρ_H) = (1/(|G||H|)) Σ_{g,h} tr(gh)
             = (|G∩H| − |G∩(−H)|) / 2ⁿ,
```

because `tr(gh)` is `±2ⁿ` only when `gh = ±I`. The **meet is a subgroup
intersection** (Gaussian elimination on the symplectic vectors), *not* the
bitwise `∩` of HLLSet supports — different lattices, different meets. The
closed form `2^{d−n}` with `d = dim(G_vec ∩ H_vec)` holds only when the two
groups are **phase-aligned**; otherwise the `−H` term can cancel the overlap.

In [4]:

def graph_generators(n, edges):
    gens = []
    for i in range(n):
        x, z = 1 << i, 0
        for (a, b) in edges:
            if a == i:
                z |= 1 << b
            if b == i:
                z |= 1 << a
        gens.append((0, x, z))
    return gens


def overlap_sq_from_groups(G, H, n):
    s = sum(pauli_trace(pauli_mul(g, h, n), n) for g in G for h in H)
    return (s / (len(G) * len(H))).real


n = 3
pairs = [
    ([], [(0, 1)]),                       # |+++> vs edge(0,1)
    ([(0, 1)], [(1, 2)]),                 # two edge graphs
    ([(0, 1), (1, 2), (0, 2)], []),       # triangle vs no edges
]
for e1, e2 in pairs:
    G = group_closure(graph_generators(n, e1), n)
    H = group_closure(graph_generators(n, e2), n)
    numeric = abs(np.vdot(graph_state(n, e1), graph_state(n, e2))) ** 2
    group_formula = overlap_sq_from_groups(G, H, n)
    gvec = [(x | (z << n)) for _, x, z in graph_generators(n, e1)]
    hvec = [(x | (z << n)) for _, x, z in graph_generators(n, e2)]
    d = gf2_intersection_dim(gvec, hvec, 2 * n)
    print("edges %s vs %s:" % (e1, e2))
    print("   numeric=%.4f  group-formula=%.4f  | subspace d=%d -> 2^(d-n)=%.4f"
          % (numeric, group_formula, d, 2.0 ** (d - n)))

# Phase caveat: |0> and |1> have the SAME vector subspace (span{Z}) but the
# generator signs differ, and the overlap is 0, not 2^{1-1}.
G0 = group_closure([(0, 0, 1)], 1)
G1 = group_closure([(2, 0, 1)], 1)         # i² Z = -Z
psi0 = np.array([1, 0], complex)
psi1 = np.array([0, 1], complex)
print("\n|0> vs |1>: numeric=%.4f  group=%.4f  (subspace meet alone would give 1.0)"
      % (abs(np.vdot(psi0, psi1)) ** 2, overlap_sq_from_groups(G0, G1, 1)))

edges [] vs [(0, 1)]:
   numeric=0.2500  group-formula=0.2500  | subspace d=1 -> 2^(d-n)=0.2500
edges [(0, 1)] vs [(1, 2)]:
   numeric=0.2500  group-formula=0.2500  | subspace d=1 -> 2^(d-n)=0.2500
edges [(0, 1), (1, 2), (0, 2)] vs []:
   numeric=0.0000  group-formula=0.0000  | subspace d=1 -> 2^(d-n)=0.2500

|0> vs |1>: numeric=0.0000  group=0.0000  (subspace meet alone would give 1.0)



## §4 — The EWM trajectory as a code filtration

The ring is exactly a binary code with Gaussian elimination. This is where the
correspondence is exact and useful:

- **pure re-basis** (same span, different insertion order) ↔ an **invertible**
  GF(2) transform: the changes of basis compose to the identity — the Clifford
  frame change;
- **extension** (a new independent generator) ↔ an isometry that raises the
  rank by one (encoding a new logical qubit), not invertible;
- **windowed eviction** ↔ discarding a generator: the rank can drop.

We replay the mirror from notebook 23 on small vectors.

In [5]:

class BoolBasis:
    def __init__(self):
        self.basis, self.pivots = [], []

    def dimension(self):
        return len(self.basis)

    def reduce(self, x):
        used = []
        while x:
            p = (x & -x).bit_length() - 1
            if p in self.pivots:
                j = self.pivots.index(p)
                x ^= self.basis[j]
                used.append(j)
            else:
                return x, used
        return 0, used

    def coordinates(self, x):
        res, used = self.reduce(x)
        if res:
            return None
        c = [False] * len(self.basis)
        for j in used:
            c[j] = not c[j]
        return c

    def insert(self, x):
        res, _ = self.reduce(x)
        if res == 0:
            return False
        p = (res & -res).bit_length() - 1
        for i in range(len(self.basis)):
            if (self.basis[i] >> p) & 1:
                self.basis[i] ^= res
        self.pivots.append(p)
        self.basis.append(res)
        return True

    def change_of_basis(self, src):
        return [self.coordinates(b) for b in src.basis]


def basis_over(sets):
    b = BoolBasis()
    for s in sets:
        b.insert(s)
    return b


gens = [0b000011, 0b000110, 0b001100, 0b011000]   # 6-bit vectors, rank 4
fwd = basis_over(gens)
rev = basis_over(list(reversed(gens)))
print("same span, two orders: dim fwd=%d rev=%d (equal: %s)"
      % (fwd.dimension(), rev.dimension(), fwd.dimension() == rev.dimension()))

MnO = rev.change_of_basis(fwd)      # forward elements in the reverse basis
Nom = fwd.change_of_basis(rev)      # reverse elements in the forward basis
k = fwd.dimension()
# transport a coordinate vector forward->reverse->forward and check identity
a = [bool(g & 1) for g in range(k)]
b = [False] * k
for j, col in enumerate(MnO):
    if a[j]:
        for i, bit in enumerate(col):
            b[i] ^= bit
a2 = [False] * k
for i2, row in enumerate(Nom):
    if b[i2]:
        for j2, bit in enumerate(row):
            a2[j2] ^= bit
print("forward->reverse->forward coordinates round-trip:", a2 == a,
      "=> the re-basis is invertible (Clifford-like)")

ext = BoolBasis()
for s in gens + [0b100000]:          # bit 5 is a genuinely new direction
    ext.insert(s)
Mext = ext.change_of_basis(fwd)      # all Some (monotone growth)
print("extension: dim %d -> %d | old columns representable %d/%d | new directions %d"
      % (k, ext.dimension(), sum(c is not None for c in Mext), k,
         ext.dimension() - k))

win = basis_over(gens[:2])
print("windowed (2 generators, monotone was %d): rank drops to %d with eviction"
      % (k, win.dimension()))

same span, two orders: dim fwd=4 rev=4 (equal: True)
forward->reverse->forward coordinates round-trip: True => the re-basis is invertible (Clifford-like)
extension: dim 4 -> 5 | old columns representable 4/4 | new directions 1
windowed (2 generators, monotone was 4): rank drops to 2 with eviction



## §5 — Where it breaks: Boolean vs. orthomodular

The decisive difference is the lattice. HLLSet operations form a **Boolean**
(distributive) lattice; the projections of a Hilbert space form an
**orthomodular**, non-distributive lattice. Concretely, in `C²` take three
distinct rank-1 projections `P, Q, R`: `Q ∨ R = I` and `Q ∧ R = 0`, so

```text
P ∧ (Q ∨ R) = P        but        (P ∧ Q) ∨ (P ∧ R) = 0 ∨ 0 = 0.
```

In [6]:

def proj(ket):
    ket = ket / np.linalg.norm(ket)
    return np.outer(ket, ket.conj())


ket0 = np.array([1, 0], complex)
ketp = np.array([1, 1], complex) / np.sqrt(2)
keti = np.array([1, 1j], complex) / np.sqrt(2)
P, Q, R = proj(ket0), proj(ketp), proj(keti)

# rank-1 distinct: meet = 0, join = identity (two distinct rays span C^2)
lhs = P @ np.eye(2)                       # P ∧ (Q ∨ R) = P ∧ I = P
rhs = np.zeros((2, 2), complex)           # (P ∧ Q) ∨ (P ∧ R) = 0
print("quantum projection lattice distributive here:",
      np.allclose(lhs, rhs), "(P != 0, but both sides differ)")
print("  P ∧ (Q∨R) rank =", np.linalg.matrix_rank(lhs),
      "| (P∧Q)∨(P∧R) rank =", np.linalg.matrix_rank(rhs))

# Boolean lattice is distributive: check on subsets of {0,1,2}
U = {0, 1, 2}
sets = [set(s) for k in range(4) for s in itertools.combinations(U, k)]
ok = all(
    A.intersection(B.union(C)) == A.intersection(B).union(A.intersection(C))
    for A in sets for B in sets for C in sets
)
print("Boolean set lattice distributive for all A,B,C:", ok)

quantum projection lattice distributive here: False (P != 0, but both sides differ)
  P ∧ (Q∨R) rank = 1 | (P∧Q)∨(P∧R) rank = 0
Boolean set lattice distributive for all A,B,C: True



## Summary

The correspondence is real, precise and useful **inside the GF(2) / stabilizer
corner**, and it breaks in a well-defined place.

Exact / structural:

- `Δ` is group multiplication; the span is a binary code; **rank = entropy**
  (verified on graph states against a numeric partial trace);
- Gaussian elimination is the stabilizer simulation core; a **pure re-basis is
  an invertible GF(2) transform = Clifford frame change**, verified by the
  coordinate round-trip; extension is an isometry, eviction irreversibly drops
  a generator;
- the monotone basis is a **filtration of codes** `S₁ ⊆ S₂ ⊆ …`;
- stabilizer **overlap** is governed by the group intersection (the meet in the
  subgroup lattice), not by the bitwise `∩` of HLLSet supports.

Analogy only (do not overclaim):

- `residual ≠ 0` is *like* non-stabilizerness ("magic"), but it is a
  non-canonical coset representative, not a monotone;
- `cover` is a classical support, not a reduced density matrix.

Breaks:

- the EWM lattice is **Boolean/distributive**; quantum projections are
  orthomodular and non-distributive (§5);
- no amplitudes, phases, interference or Born rule (the Pauli phase/symplectic
  central charge is absent unless added — §3 shows a case where the vector
  subspace alone is not enough);
- a plain GF(2) span is a binary code, not a stabilizer group: to make it
  literal, fix a symplectic form and restrict to isotropic/Lagrangian
  subspaces.